In [ ]:
from pathlib import Path

DATA = Path("../data/raw/model_build_inputs")

for file in DATA.iterdir():
    print(file.name, "->", round(file.stat().st_size / (1024**2), 2), "MB")

In [ ]:
import json
from pathlib import Path
from collections import Counter

DATA = Path("../data/raw/model_build_inputs")

with open(DATA / "package_data.json", "r", encoding="utf-8") as f:
    package_data = json.load(f)

with open(DATA / "route_data.json", "r", encoding="utf-8") as f:
    route_data = json.load(f)

# ---------- PACKAGE ANALYSIS ----------

status_counts = Counter()
total_packages = 0
total_stops_package = 0
missing_windows = 0
service_times = []

for route_id, stops in package_data.items():
    for stop_id, packages in stops.items():
        total_stops_package += 1

        for package_id, pkg in packages.items():
            total_packages += 1

            status_counts[pkg.get("scan_status")] += 1

            tw = pkg.get("time_window", {})
            if not tw.get("start_time_utc") or not tw.get("end_time_utc"):
                missing_windows += 1

            if pkg.get("planned_service_time_seconds") is not None:
                service_times.append(
                    pkg["planned_service_time_seconds"]
                )

print("=== PACKAGE DATA ===")
print("Routes:", len(package_data))
print("Total packages:", total_packages)
print("Total package stops:", total_stops_package)
print("Scan statuses:", status_counts)
print("Missing time windows:", missing_windows)
print("Average service time:",
      sum(service_times) / len(service_times))


# ---------- ROUTE ANALYSIS ----------

stop_types = Counter()
total_stops = 0
stations = Counter()
zones = Counter()

for route_id, route in route_data.items():

    stations[route.get("station_code")] += 1

    for stop_id, stop in route.get("stops", {}).items():
        total_stops += 1
        stop_types[stop.get("type")] += 1

        if stop.get("zone_id"):
            zones[stop.get("zone_id")] += 1

print("\n=== ROUTE DATA ===")
print("Routes:", len(route_data))
print("Total stops:", total_stops)
print("Stop types:", stop_types)
print("Stations:", len(stations))
print("Zones:", len(zones))

In [ ]:
print("\n=== SAMPLE ROUTE ===")
route_id = next(iter(route_data))
print(route_id)

print("\nRoute fields:")
print(route_data[route_id].keys())

print("\nPackage stop codes:")
print(list(package_data[route_id].keys())[:10])

print("\nRoute stop codes:")
print(list(route_data[route_id]["stops"].keys())[:10])

In [ ]:
print("\n=== DATA QUALITY ===")

print("Package routes missing from route_data:",
      len(set(package_data) - set(route_data)))

print("Route routes missing from package_data:",
      len(set(route_data) - set(package_data)))